In [ ]:
import torch
from torch.utils.data import DataLoader, random_split
from torch.optim import Adam
from torchvision import transforms
from info_nce import InfoNCE

from tqdm.auto import tqdm

import sqlite3
import pandas as pd

from pathlib import Path
import sys

ROOT = Path.cwd().parents[1]
sys.path.append(str(ROOT))

EMBED_PATH = ROOT / "data/embeddings/cont_embeds.pt"
EMBED_NAME = EMBED_PATH.stem

RESULTS_DIR = ROOT / "data/results" / EMBED_NAME

DATA_DIR = ROOT / "data"
SQL_DIR = DATA_DIR / "sql"
DB_PATH = SQL_DIR / "metadata.db"

MODEL_DIR = ROOT / "models"

In [ ]:
contrastive_transform = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2,
        hue=0.1,
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

In [3]:
conn = sqlite3.connect(DB_PATH)

df = pd.read_sql_query("""
    SELECT path, split
    FROM meta
""", conn)

conn.close()

train_paths = df[df["split"] == "train"]["path"].to_list()
test_paths = df[df["split"] != "train"]["path"].to_list()

In [4]:
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Device: {device}")

Device: cpu


In [5]:
dino = torch.hub.load(
            "facebookresearch/dinov2",
            "dinov2_vits14"
        )

dino.to(device)
dino.eval()

for p in dino.parameters():
    p.requires_grad = False

Using cache found in C:\Users\niall/.cache\torch\hub\facebookresearch_dinov2_main
C:\Users\niall/.cache\torch\hub\facebookresearch_dinov2_main\dinov2\layers\swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
C:\Users\niall/.cache\torch\hub\facebookresearch_dinov2_main\dinov2\layers\attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
C:\Users\niall/.cache\torch\hub\facebookresearch_dinov2_main\dinov2\layers\block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


In [15]:
%load_ext autoreload
%autoreload 2

from src.fine_tune import ModelData, ProjectionHead, train_one_epoch, evaluate


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
BATCH_SIZE = 32
NUM_WORKERS = 8
PIN_MEMORY = device == "cuda"

In [ ]:
dataset = ModelData(train_paths, root=ROOT, transform=contrastive_transform)
test_set = ModelData(test_paths, root=ROOT, transform=test_transform)

train_split = int(len(dataset) * 0.95)
val_split = len(dataset) - train_split

generator = torch.Generator().manual_seed(42)

train_set, val_set = random_split(
    dataset,
    [train_split, val_split],
    generator=generator
)

train_loader = DataLoader(
        train_set,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY
    )

val_loader = DataLoader(
    val_set,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY
)

test_loader = DataLoader(
    test_set,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY
)

In [18]:
epochs = 2

model = ProjectionHead(dim=384).to(device)

optimizer = Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
criterion = InfoNCE()

In [ ]:
losses = {"train" : [], "val" : []}

for epoch in tqdm(range(epochs)):
    train_loss = train_one_epoch(model, dino, train_loader, criterion, optimizer, device)
    val_loss = evaluate(model, dino, val_loader, criterion, device)

    losses["train"].append(train_loss)
    losses["eval"].append(val_loss)

    print("Train Loss:", train_loss)
    print("Eval Loss:", val_loss)
  

Train Loss: 0.7748191096542174
Eval Loss: 1.882391573102386
Train Loss: 0.6449837893770453
Eval Loss: 1.8222775045368407
